In [20]:
!pip install importnb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.4 MB/s eta 0:00:00


In [21]:
import sys
from importnb import Notebook

# Tell Python where the folder is (change this to your actual folder path)
sys.path.append("/content/drive/MyDrive/Colab Notebooks/Lora/")

# Now import it like a normal library
with Notebook():
    import Utils

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.6/419.6 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # Your exact test model
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters (This is the "surgical upgrade" part)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The Rank. 16 is great for 150 documents.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
from datasets import load_dataset

# 1. Load your uploaded file
dataset = load_dataset("json", data_files={"train": r"/content/drive/MyDrive/Colab Notebooks/Lora/training_set.jsonl"}, split="train")

# 2. Map the data into the Llama 3.2 Chat Template
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # This matches the Llama-3.2 Instruct format exactly
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{instruction}<|eot_id|>" \
               f"<|start_header_id|>user<|end_header_id|>\n\n{input}<|eot_id|>" \
               f"<|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80, # Seeing 150 docs once or twice
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# START THE TRAINING
trainer_stats = trainer.train()


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/150 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 150 | Num Epochs = 5 | Total steps = 80
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.820845
2,1.880712
3,2.169000
4,1.951177
5,1.693651
6,1.725137
7,1.890970
8,1.564979
9,1.505369
10,1.665536


In [ ]:
FastLanguageModel.for_inference(model)

def extract_pico(trial_text):
    instruction = "Extract the Participants, Intervention, and Outcome from the trial text. Return the result strictly as a JSON object."

    # Format the prompt exactly like the training data
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": trial_text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Generate the extraction
    outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,        # Lower temperature = more focused/less creative
    repetition_penalty = 1.2 )

    # Decode and clean up the result
    # response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # json_part = response.split("assistant")[-1].strip()
    # return json_part
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_string = response.split("assistant")[-1].strip()
    try:
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Malformed JSON string: {json_string}")
        return {}

# --- TEST IT WITH A NEW TRIAL ---
test_text = " Pre-operative short-term pulmonary rehabilitation for patients of chronic obstructive pulmonary disease undergoing coronary artery bypass graft surgery. The role of pre-operative short-term pulmonary rehabilitation in patients with chronic obstructive pulmonary disease who undergo coronary artery bypass graft surgery has been assessed for the first time prospectively. Forty-five patients posted for coronary artery bypass graft surgery were randomised to receive either short-term pulmonary rehabilitation (group I) or no such programme (group II). Patients of both the groups were evenly matched with respect to age, sex, body surface area, duration and severity of chronic obstructive pulmonary disease and coronary artery disease. Normal individuals who evenly matched with the study group were assessed for normal respiratory function parameters. Pre-operative and post-operative peak expiratory flow rate, inspiratory capacity, post-operative ventilation time, post-operative pulmonary complication and hospital stay were determined in both the groups. Peak expiratory flow rate (220.0 +/- 12.9 and 324.3 +/- 84.3 in group I, 218.0 +/- 16.4 and 260.5 +/- 35.2 in group II) and inspiratory capacity (844.0 +/- 147.4 and 1100.0 +/- 158.1 in group I, 830.0 +/- 117.4 and 1090 +/- 137 in group II) were significantly lower before and after surgery respectively in both groups compared to normal values. Even though both groups showed a significant rise in post-operative peak expiratory flow rate and inspiratory capacity after surgery, the post-operative peak expiratory flow rate and inspiratory capacity in group I was significantly higher than in group II. In group I, the post-operative ventilation time (24.5 +/- 6.00 hours), post-operative complications (n = 4) and hospital stay (12.4 +/- 3.6 days) were significantly lower than in group II (35.2 +/- 22.3 hours, n = 11, 18.8 +/- 6.6 days respectively). These data suggest that short-term pulmonary rehabilitation is feasible and effective in improving pulmonary functions before and after surgery and in reducing surgical morbidity and cost of medical care significantly."
print(extract_pico(test_text))

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"Participants": ["chronic obstructive pulmonary disease", "coronary artery bypass graft surgery."], "Intervention": ["short-term pulmonary rehabilitation", "no such programme", "pulmonary rehabilitation", "No such programme", "Pulmonary rehabilitation", "Short-term pulmonary rehabilitation", "No such programme", "Pulmonary rehabilitation", "Short-term pulmonary rehabilitation", "No such programme, Pulmonary rehabilitation", "Pulmonary rehabilitation"], "Outcome": ["peak expiratory flow rate", "inspiratory capacity", "post-operative ventilation time", "post-operative pulmonary complication", "hospital stay"]}


In [ ]:

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

import shutil
shutil.copytree("lora_model", r"/content/drive/MyDrive/Colab Notebooks/Lora/medical_lora_model")


'/content/drive/MyDrive/Colab Notebooks/Lora/medical_lora_model'

In [7]:
from unsloth import FastLanguageModel

# Direct loading from your Drive folder
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/Colab Notebooks/Lora/medical_lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Switch to inference mode
FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.4.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [9]:
import json

In [18]:
FastLanguageModel.for_inference(model)

def extract(trial_text):
    instruction = "Extract the Participants, Intervention, and Outcome from the trial text. Return the result strictly as a JSON object."

    # Format the prompt exactly like the training data
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": trial_text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Generate the extraction
    outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,        # Lower temperature = more focused/less creative
    repetition_penalty = 1.2 )

    # Decode and clean up the result
    # response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # json_part = response.split("assistant")[-1].strip()
    # return json_part
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_string = response.split("assistant")[-1].strip()
    try:
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Malformed JSON string: {json_string}")
        return {}



In [19]:
file_path = r"/content/drive/MyDrive/Colab Notebooks/Lora/result_lora.csv"




In [22]:
import pandas as pd
df = pd.read_excel(r"/content/drive/MyDrive/Colab Notebooks/cwk_data/cwk_train_data.xlsx")

In [24]:
Utils.config(df, file_path, 200, extract)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error decoding JSON: Unterminated string starting at: line 1 column 1472 (char 1471)
Malformed JSON string: {"Participants": ["Helicobacter pylori"], "Intervention": ["H2 blockaders", "ranitidine", "lansoprazole", "antisecretory drugs", "r-anitidine", "clarithromycin, metrinidazole.", "PPI", "ranitidine", "Lansoprazole", "ranitidine", "metrinidazole", "clarithromycin", "ranitidine", "Metronidazole", "Clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithromycin", "ranitidine", "Metronidazole", "clarithr

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished and saved Doc ID: 10037531


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Error decoding JSON: Unterminated string starting at: line 1 column 1101 (char 1100)
Malformed JSON string: {"Participants": ["small-cell", "lung"], "Intervention": ["teniposide", "whole-brain radiotherapy", "Teniposide", "Whole-brain radiotherapy", "chemotherapy", "teniposide", "WBRT.", "WBRT", "teniposide", "WBRT", "WBRT", "teniposide", "WBRT", "WBRT", "teniposide", "WBRT,", "WBRT", "WBRT", "WBRT", "teniposide", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "teniposide", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", "WBRT", 

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finished and saved Doc ID: 11013775


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Error decoding JSON: Unterminated string starting at: line 1 column 1942 (char 1941)
Malformed JSON string: {"Participants": ["long-term", "state hospital"], "Intervention": ["clozapine", "conventional antipsychotic medication", "open-label, randomized controlled trial", "physicians' choice medications", "clozapine", "conventional antipsychotic medications", "Clozapine", "usual care", "antipsychotic medications", "medications", "service utilization", "other costs.", "both", "usual care", "unusual care", "conventional antipsychotic medications", "clozapine", "conventional antipsychotic medications", "Clozapine", "usual care", "unusual care", "conventional antipsychotic medications", "clozapine", "conventional antipsychotic medications", "Clozapine", "usual care", "unusual care", "conventional antipsychotic medications", "clozapine", "conventional antipsychotic medications", "Clozapine", "usual care", "unusual care", "conventional antipsychotic medications", "clozapine", "conventional an